In [6]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.utils.features import split_features

In [7]:
DATA_DIR = Path("../datasets/final/ml")

DATASET_NAMES = [
    "cicids2017",
    "unsw_nb15",
    "iot23",
]

RANDOM_STATE = 1
N_SPLITS = 2

In [8]:
def build_logistic_pipeline(numerical_features, categorical_features):
    preprocessor = ColumnTransformer(
        transformers=[
            ("numerical", StandardScaler(), numerical_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(
                solver="saga",
                max_iter=1000,
                random_state=RANDOM_STATE,
            )),
        ]
    )

In [9]:
def evaluate_training_dataset(dataset_name):
    df = pd.read_parquet(
        DATA_DIR / f"{dataset_name}_train.parquet"
    )

    numerical_features, categorical_features = split_features(df)
    feature_columns = numerical_features + categorical_features

    features = df[feature_columns]
    target = df["label_binary"].eq("ATTACK").astype(int)

    cross_validation = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    scores = cross_validate(
        build_logistic_pipeline(
            numerical_features,
            categorical_features,
        ),
        features,
        target,
        cv=cross_validation,
        scoring={
            "precision_attack": make_scorer(
                precision_score,
                pos_label=1,
                zero_division=0,
            ),
            "recall_attack": make_scorer(
                recall_score,
                pos_label=1,
                zero_division=0,
            ),
            "f1_attack": make_scorer(f1_score, pos_label=1),
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
        },
        n_jobs=1,
    )

    return {
        "dataset": dataset_name,
        "numerical_features": len(numerical_features),
        "categorical_features": len(categorical_features),
        "precision_attack_mean": scores["test_precision_attack"].mean(),
        "precision_attack_std": scores["test_precision_attack"].std(),
        "recall_attack_mean": scores["test_recall_attack"].mean(),
        "recall_attack_std": scores["test_recall_attack"].std(),
        "f1_attack_mean": scores["test_f1_attack"].mean(),
        "f1_attack_std": scores["test_f1_attack"].std(),
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
        "balanced_accuracy_std": scores["test_balanced_accuracy"].std(),
    }

In [10]:
results = [
    evaluate_training_dataset(dataset_name)
    for dataset_name in DATASET_NAMES
]

results_df = pd.DataFrame(results)

display(results_df)

C:\tcc\network-ids-ml-generalization\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\tcc\network-ids-ml-generalization\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\tcc\network-ids-ml-generalization\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\tcc\network-ids-ml-generalization\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\tcc\network-ids-ml-generalization\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\tcc\network-ids-m

,dataset,numerical_features,categorical_features,precision_attack_mean,precision_attack_std,recall_attack_mean,recall_attack_std,f1_attack_mean,f1_attack_std,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std
0,cicids2017,16,4,0.914057,0.002239,0.911313,0.001125,0.912680,0.000552,0.965124,0.000285,0.944945,0.000244
1,unsw_nb15,16,4,0.893227,0.000809,0.850843,0.004735,0.871515,0.002869,0.949828,0.001006,0.912709,0.002405
2,iot23,16,4,0.990007,0.000908,0.990658,0.000467,0.990332,0.000688,0.996131,0.000276,0.994079,0.000348
